## r/bakeoff subreddit data cleaning and enriching

This notebook cleans the raw r/baking data, inclduing:
- Dropping unneeded columns
- Converting the post date from UTC (ms since epoch) to MM/DD/YYYY
- Doing a fuzzy match to filter for posts that are similar to GBBO technical bakes
- Calculating and adding a column that denotes the distance from the relevant GBBO episode in which the post was made.

In [15]:
import os
import json
import glob
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz, utils
from pathlib import Path
from datetime import datetime, timezone
import re

In [16]:
data_csv = Path("/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/bakoff_reddit_data_raw.csv")
if data_csv.is_file():
    bakeoff_reddit_df = pd.read_csv(data_csv)
    print(f"file already exists")
else:
    data=[]
    with open ('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/r_bakeoff_posts.jsonl', 'r') as file:
        for line in file:
            data.append(json.loads(line))
        bakeoff_reddit_df = pd.read_json("/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/r_bakeoff_posts.jsonl", lines=True)

bakeoff_reddit_df

,archived,author,author_flair_css_class,author_flair_text,created,created_utc,distinguished,domain,downs,edited,...,event_start,call_to_action,author_is_blocked,_meta,previous_selftext,location_lat,location_long,location_name,websocket_url,outbound_link
0,1.0,Dafman,NaN,NaN,1.407411e+09,1407407446,NaN,metro.co.uk,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,Brickie78,NaN,NaN,1.407412e+09,1407408524,NaN,self.bakeoff,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.0,Dafman,NaN,NaN,1.407488e+09,1407484582,NaN,self.bakeoff,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0,Brickie78,NaN,NaN,1.407589e+09,1407585563,NaN,buzzfeed.com,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.0,Brickie78,NaN,NaN,1.408009e+09,1408005425,NaN,s3-ec.buzzfed.com,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7638,0.0,dlelovcit,NaN,NaN,1.782491e+09,1782490589,NaN,i.redd.it,0.0,0,...,NaN,NaN,0.0,{'retrieved_2nd_on': 1782620211},NaN,NaN,NaN,NaN,wss://k8s-lb.wss.redditmedia.com/link/1ugbbia?...,"{'created': None, 'expiration': None, 'url': '..."
7639,0.0,applejam99,NaN,NaN,1.782550e+09,1782549913,NaN,self.bakeoff,0.0,0,...,NaN,NaN,0.0,"{'is_edited': True, 'retrieved_2nd_on': 178267...",NaN,NaN,NaN,NaN,wss://k8s-lb.wss.redditmedia.com/link/1ugxhpx?...,NaN
7640,0.0,TritiyaPandav,NaN,NaN,1.783960e+09,1783959646,NaN,i.redd.it,0.0,0,...,NaN,NaN,0.0,{'retrieved_2nd_on': 1784089265},NaN,NaN,NaN,NaN,wss://k8s-lb.wss.redditmedia.com/link/1uvg847?...,"{'created': None, 'expiration': None, 'url': '..."
7641,0.0,Babbettee,NaN,NaN,1.784036e+09,1784036409,NaN,i.imgur.com,0.0,0,...,NaN,NaN,0.0,{'retrieved_2nd_on': 1784166017},NaN,NaN,NaN,NaN,wss://k8s-lb.wss.redditmedia.com/link/1uw90gk?...,"{'created': 1784036427000, 'expiration': 17840..."


In [17]:
bakeoff_reddit_df_cols_to_keep = [
    'created_utc',
    'thumbnail',
    'title',
    'url',
]

bakeoff_reddit_df_clean = bakeoff_reddit_df[bakeoff_reddit_df_cols_to_keep]
bakeoff_reddit_df_clean.to_csv('bakeoff_reddit_data_complete.csv')
bakeoff_reddit_df_clean

,created_utc,thumbnail,title,url
0,1407407446,http://b.thumbs.redditmedia.com/l8zXJUWSLOeuDM...,Bake Off: Who left the competition first?,http://metro.co.uk/2014/08/06/great-british-ba...
1,1407408524,self,Can we talk about that beard?,http://www.reddit.com/r/bakeoff/comments/2cvg9...
2,1407484582,self,Week 1 Discussion Thread,http://www.reddit.com/r/bakeoff/comments/2cyoa...
3,1407585563,http://b.thumbs.redditmedia.com/rxTXn_LGsyKci0...,"Claire responds to ""Fat Shaming"" tweets",http://www.buzzfeed.com/robynwilder/fat-shamin...
4,1408005425,http://b.thumbs.redditmedia.com/buMonV7OYVqoQw...,The Mary Berry Death Stare,http://s3-ec.buzzfed.com/static/2014-08/13/16/...
...,...,...,...,...
7638,1782490589,https://preview.redd.it/v43x7yfmjn9h1.jpeg?wid...,Those eyes...,https://i.redd.it/v43x7yfmjn9h1.jpeg
7639,1782549913,self,Best & worst country spin-offs?,https://www.reddit.com/r/bakeoff/comments/1ugx...
7640,1783959646,https://preview.redd.it/hvvspfkxv0dh1.png?widt...,"Baked with love, served with happiness. 🍰✨ Eve...",https://i.redd.it/hvvspfkxv0dh1.png
7641,1784036409,https://external-preview.redd.it/Y3bq0nLFkK560...,Found Prue in my kids legos...,https://i.imgur.com/JNR0j22.jpeg


In [18]:
bakeoff_reddit_df_clean['date'] = pd.to_datetime(bakeoff_reddit_df_clean['created_utc'], unit='s')
bakeoff_reddit_df_clean['date'] = bakeoff_reddit_df_clean['date'].dt.strftime('%m/%d/%Y')
bakeoff_reddit_df_clean

,created_utc,thumbnail,title,url,date
0,1407407446,http://b.thumbs.redditmedia.com/l8zXJUWSLOeuDM...,Bake Off: Who left the competition first?,http://metro.co.uk/2014/08/06/great-british-ba...,08/07/2014
1,1407408524,self,Can we talk about that beard?,http://www.reddit.com/r/bakeoff/comments/2cvg9...,08/07/2014
2,1407484582,self,Week 1 Discussion Thread,http://www.reddit.com/r/bakeoff/comments/2cyoa...,08/08/2014
3,1407585563,http://b.thumbs.redditmedia.com/rxTXn_LGsyKci0...,"Claire responds to ""Fat Shaming"" tweets",http://www.buzzfeed.com/robynwilder/fat-shamin...,08/09/2014
4,1408005425,http://b.thumbs.redditmedia.com/buMonV7OYVqoQw...,The Mary Berry Death Stare,http://s3-ec.buzzfed.com/static/2014-08/13/16/...,08/14/2014
...,...,...,...,...,...
7638,1782490589,https://preview.redd.it/v43x7yfmjn9h1.jpeg?wid...,Those eyes...,https://i.redd.it/v43x7yfmjn9h1.jpeg,06/26/2026
7639,1782549913,self,Best & worst country spin-offs?,https://www.reddit.com/r/bakeoff/comments/1ugx...,06/27/2026
7640,1783959646,https://preview.redd.it/hvvspfkxv0dh1.png?widt...,"Baked with love, served with happiness. 🍰✨ Eve...",https://i.redd.it/hvvspfkxv0dh1.png,07/13/2026
7641,1784036409,https://external-preview.redd.it/Y3bq0nLFkK560...,Found Prue in my kids legos...,https://i.imgur.com/JNR0j22.jpeg,07/14/2026


In [19]:
bakeoff_reddit_post_titles = bakeoff_reddit_df_clean['title']
bakeoff_reddit_post_titles

0               Bake Off: Who left the competition first?
1                           Can we talk about that beard?
2                                Week 1 Discussion Thread
3                 Claire responds to "Fat Shaming" tweets
4                              The Mary Berry Death Stare
                              ...                        
7638                                        Those eyes...
7639                      Best & worst country spin-offs?
7640    Baked with love, served with happiness. 🍰✨ Eve...
7641                       Found Prue in my kids legos...
7642                 Interesting news for those in the US
Name: title, Length: 7643, dtype: str

In [20]:
technicals_df = pd.read_csv('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/TechnicalBakes.csv')
technicals_cols = technicals_df['Technical']
technicals_cols

0                       victoria sandwich
1                                  scones
2                                     cob
3                 mini hot lemon soufflés
4                         cornish pasties
                      ...                
129          lemon and thyme drizzle cake
130    orange and ginger treacle puddings
131                      caterpiller cake
132                       tart aux pommes
133                     lardy cake slices
Name: Technical, Length: 134, dtype: str

In [21]:
#iterate through each technical
matched_csv = Path("/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/bakeoff_reddit_posts_matched.csv")

if matched_csv.is_file():
    bakeoff_reddit_df_clean = pd.read_csv(matched_csv)
    print(f"file already exists")

else:

    choices = technicals_cols
    titles = bakeoff_reddit_df_clean['title'].tolist()

    score_matrix = process.cdist(titles, choices, scorer=fuzz.ratio)

    best_scores = score_matrix.max(axis=1)
    best_idx = score_matrix.argmax(axis=1)

    bakeoff_reddit_df_clean['bakeoff_score'] = best_scores
    bakeoff_reddit_df_clean['bakeoff_match'] = [choices[i] for i in best_idx]
    bakeoff_reddit_df_clean['bakeoff'] = best_scores > 70

    bakeoff_reddit_df_clean = bakeoff_reddit_df_clean[bakeoff_reddit_df_clean['bakeoff']][['date','title', 'bakeoff_match', 'bakeoff_score']]
    bakeoff_reddit_df_clean.to_csv('bakeoff_reddit_posts_matched.csv')

In [22]:
#add air dates
technical_bakes = pd.read_csv('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/TechnicalBakes.csv')

bakeoff_reddit_df_clean_dated = pd.merge(bakeoff_reddit_df_clean, technical_bakes, left_on='bakeoff_match', right_on='Technical', how='left')
bakeoff_reddit_df_clean_dated


,date,title,bakeoff_match,bakeoff_score,Airdate,Season,Episode,Theme,Signature,Signature Time (min),Technical,Technical Time (min),Showstopper,Showstopper Time (min),challenge level numeric
0,09/01/2016,I made Viennese whirls,viennese whirls,78.947365,8/31/2016,7,2,biscuit,24 iced biscuits,150.0,viennese whirls,90.0,gingerbread 3d scene,240.0,2.0
1,09/04/2016,I Made Viennese Whirls!,viennese whirls,71.794868,8/31/2016,7,2,biscuit,24 iced biscuits,150.0,viennese whirls,90.0,gingerbread 3d scene,240.0,2.0
2,09/11/2016,Dampfnudel!,dampfnudel,85.714287,9/7/2016,7,3,bread,chocolate loaf,150.0,dampfnudel,120.0,savoury plaited centrepiece,240.0,NaN
3,10/15/2016,Jumbles!,jumbles,75.000000,10/12/2016,7,8,tudor,shaped savoury pie,180.0,jumbles,90.0,3d marchpane cake,210.0,NaN
4,10/24/2016,Made fondant fancies,fondant fancies,88.888885,10/16/2012,3,10,final,pithivier,150.0,fondant fancies,150.0,chiffon cake,240.0,2.0
5,12/29/2018,I made the 8 strand plaited loaf!,eight-strand plaited loaf,72.413795,8/21/2012,3,2,bread,12 flatbreads,150.0,eight-strand plaited loaf,120.0,12 sweet and 12 savoury bagels,240.0,NaN
6,05/18/2019,I made Queen of Puddings,queen of puddings,73.170731,9/18/2012,3,6,pudding,2 sponge puddings,120.0,queen of puddings,NaN,strudel,210.0,2.0
7,09/17/2019,Paul’s Chocolate Teacakes!,chocolate teacakes,75.555557,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0
8,08/16/2020,Aebleskiver!,æbleskiver,78.260872,10/16/2018,9,8,danish,2 smørrebrød,210.0,æbleskiver,60.0,kagemand/kagekone,270.0,1.0
9,09/13/2020,My lemon meringue pie!,lemon meringue pie,90.000000,10/11/2022,13,5,desserts,8 steamed puddings,120.0,lemon meringue pie,120.0,hidden surprise mousse dessert,270.0,2.0


In [23]:
bakeoff_reddit_df_clean_dated.drop(columns=['bakeoff_match'])
bakeoff_reddit_df_clean_dated['reddit_post_title'] = bakeoff_reddit_df_clean_dated['title']
bakeoff_reddit_df_clean_dated['reddit_post_date'] = bakeoff_reddit_df_clean_dated['date']
bakeoff_reddit_df_clean_dated

,date,title,bakeoff_match,bakeoff_score,Airdate,Season,Episode,Theme,Signature,Signature Time (min),Technical,Technical Time (min),Showstopper,Showstopper Time (min),challenge level numeric,reddit_post_title,reddit_post_date
0,09/01/2016,I made Viennese whirls,viennese whirls,78.947365,8/31/2016,7,2,biscuit,24 iced biscuits,150.0,viennese whirls,90.0,gingerbread 3d scene,240.0,2.0,I made Viennese whirls,09/01/2016
1,09/04/2016,I Made Viennese Whirls!,viennese whirls,71.794868,8/31/2016,7,2,biscuit,24 iced biscuits,150.0,viennese whirls,90.0,gingerbread 3d scene,240.0,2.0,I Made Viennese Whirls!,09/04/2016
2,09/11/2016,Dampfnudel!,dampfnudel,85.714287,9/7/2016,7,3,bread,chocolate loaf,150.0,dampfnudel,120.0,savoury plaited centrepiece,240.0,NaN,Dampfnudel!,09/11/2016
3,10/15/2016,Jumbles!,jumbles,75.000000,10/12/2016,7,8,tudor,shaped savoury pie,180.0,jumbles,90.0,3d marchpane cake,210.0,NaN,Jumbles!,10/15/2016
4,10/24/2016,Made fondant fancies,fondant fancies,88.888885,10/16/2012,3,10,final,pithivier,150.0,fondant fancies,150.0,chiffon cake,240.0,2.0,Made fondant fancies,10/24/2016
5,12/29/2018,I made the 8 strand plaited loaf!,eight-strand plaited loaf,72.413795,8/21/2012,3,2,bread,12 flatbreads,150.0,eight-strand plaited loaf,120.0,12 sweet and 12 savoury bagels,240.0,NaN,I made the 8 strand plaited loaf!,12/29/2018
6,05/18/2019,I made Queen of Puddings,queen of puddings,73.170731,9/18/2012,3,6,pudding,2 sponge puddings,120.0,queen of puddings,NaN,strudel,210.0,2.0,I made Queen of Puddings,05/18/2019
7,09/17/2019,Paul’s Chocolate Teacakes!,chocolate teacakes,75.555557,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0,Paul’s Chocolate Teacakes!,09/17/2019
8,08/16/2020,Aebleskiver!,æbleskiver,78.260872,10/16/2018,9,8,danish,2 smørrebrød,210.0,æbleskiver,60.0,kagemand/kagekone,270.0,1.0,Aebleskiver!,08/16/2020
9,09/13/2020,My lemon meringue pie!,lemon meringue pie,90.000000,10/11/2022,13,5,desserts,8 steamed puddings,120.0,lemon meringue pie,120.0,hidden surprise mousse dessert,270.0,2.0,My lemon meringue pie!,09/13/2020


In [24]:
GBBO_bakeoff_reddit_posts_merged = bakeoff_reddit_df_clean_dated.drop(columns=['title', 'date'])
GBBO_bakeoff_reddit_posts_merged

,bakeoff_match,bakeoff_score,Airdate,Season,Episode,Theme,Signature,Signature Time (min),Technical,Technical Time (min),Showstopper,Showstopper Time (min),challenge level numeric,reddit_post_title,reddit_post_date
0,viennese whirls,78.947365,8/31/2016,7,2,biscuit,24 iced biscuits,150.0,viennese whirls,90.0,gingerbread 3d scene,240.0,2.0,I made Viennese whirls,09/01/2016
1,viennese whirls,71.794868,8/31/2016,7,2,biscuit,24 iced biscuits,150.0,viennese whirls,90.0,gingerbread 3d scene,240.0,2.0,I Made Viennese Whirls!,09/04/2016
2,dampfnudel,85.714287,9/7/2016,7,3,bread,chocolate loaf,150.0,dampfnudel,120.0,savoury plaited centrepiece,240.0,NaN,Dampfnudel!,09/11/2016
3,jumbles,75.000000,10/12/2016,7,8,tudor,shaped savoury pie,180.0,jumbles,90.0,3d marchpane cake,210.0,NaN,Jumbles!,10/15/2016
4,fondant fancies,88.888885,10/16/2012,3,10,final,pithivier,150.0,fondant fancies,150.0,chiffon cake,240.0,2.0,Made fondant fancies,10/24/2016
5,eight-strand plaited loaf,72.413795,8/21/2012,3,2,bread,12 flatbreads,150.0,eight-strand plaited loaf,120.0,12 sweet and 12 savoury bagels,240.0,NaN,I made the 8 strand plaited loaf!,12/29/2018
6,queen of puddings,73.170731,9/18/2012,3,6,pudding,2 sponge puddings,120.0,queen of puddings,NaN,strudel,210.0,2.0,I made Queen of Puddings,05/18/2019
7,chocolate teacakes,75.555557,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0,Paul’s Chocolate Teacakes!,09/17/2019
8,æbleskiver,78.260872,10/16/2018,9,8,danish,2 smørrebrød,210.0,æbleskiver,60.0,kagemand/kagekone,270.0,1.0,Aebleskiver!,08/16/2020
9,lemon meringue pie,90.000000,10/11/2022,13,5,desserts,8 steamed puddings,120.0,lemon meringue pie,120.0,hidden surprise mousse dessert,270.0,2.0,My lemon meringue pie!,09/13/2020


In [25]:
#calculating days since technical airdate in which reddit post was made

GBBO_bakeoff_reddit_posts_merged['Airdate'] = pd.to_datetime(GBBO_bakeoff_reddit_posts_merged['Airdate'])
GBBO_bakeoff_reddit_posts_merged['reddit_post_date'] = pd.to_datetime(GBBO_bakeoff_reddit_posts_merged['reddit_post_date'])

# GBBO_reddit_posts_merged['days_since_air'] = []

posts = GBBO_bakeoff_reddit_posts_merged['reddit_post_title']

for post in posts:
    GBBO_bakeoff_reddit_posts_merged['days_since_air'] = GBBO_bakeoff_reddit_posts_merged['Airdate'] - GBBO_bakeoff_reddit_posts_merged['reddit_post_date']

GBBO_bakeoff_reddit_posts_merged.to_csv('GBBO_bakeoff_reddit_posts_merged.csv', index=False)